Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence

# Báo cáo Bài tập tuần - Buổi 5
**Nguyễn Đức Phát - 24110296**

---

## 1. Vấn đề bài toán (8-Puzzle Problem)
8-Puzzle là bài toán trò chơi sắp xếp số. Trò chơi gồm một bảng 3x3 chứa 8 ô số (từ 1 đến 8) và 1 ô trống (biểu diễn bằng số 0). Mục tiêu là di chuyển ô trống (lên, xuống, sang trái, sang phải) từ một trạng thái bắt đầu (Start State) đến một trạng thái đích (Goal State) cho trước sao cho số bước di chuyển là tối ưu.

## 2. Ý tưởng của thuật toán
Bài tập này triển khai cả hai thuật toán tìm kiếm mù cơ bản trong Trí tuệ nhân tạo:
- **Breadth-First Search (BFS)**:
  - **Ý tưởng**: Sử dụng cấu trúc hàng đợi (Queue - FIFO) để duyệt qua tất cả các nút ở mức độ sâu hiện tại trước khi đi xuống mức sâu hơn.
  - **Trực quan hóa**: Tập biên (Frontier) được hiển thị theo thứ tự từ đầu hàng đợi (Head) đến cuối hàng đợi (Rear).
- **Depth-First Search (DFS)**:
  - **Ý tưởng**: Sử dụng cấu trúc ngăn xếp (Stack - LIFO) để đi sâu nhất có thể dọc theo mỗi nhánh trước khi quay lui (backtrack).
  - **Trực quan hóa**: Tập biên (Frontier) được hiển thị theo thứ tự từ đáy ngăn xếp (Bottom) lên đỉnh ngăn xếp (Top).

Cả hai thuật toán đều được cài đặt dưới dạng **Python Generator (`yield`)**, cho phép cập nhật giao diện Tkinter theo thời gian thực (Real-time Visualization) sau mỗi bước duyệt mà không gây block/đơ UI.

## 3. Độ phức tạp thuật toán
- **BFS**:
  - **Độ phức tạp thời gian**: $O(b^d)$, trong đó $b$ là hệ số nhánh (tối đa là 4), $d$ là độ sâu của lời giải.
  - **Độ phức tạp không gian**: $O(b^d)$ (cần lưu toàn bộ cây tìm kiếm trong bộ nhớ). BFS luôn đảm bảo tìm thấy đường đi ngắn nhất (tối ưu).
- **DFS**:
  - **Độ phức tạp thời gian**: $O(b^m)$, trong đó $m$ là độ sâu tối đa của cây tìm kiếm. Trong trường hợp xấu nhất, DFS có thể duyệt qua rất nhiều trạng thái trước khi tìm ra lời giải.
  - **Độ phức tạp không gian**: $O(b \cdot m)$, tiết kiệm bộ nhớ hơn BFS rất nhiều. DFS không đảm bảo tìm thấy đường đi ngắn nhất (không tối ưu).

## 4. Hướng dẫn chạy chương trình
1. Đảm bảo bạn đã cài đặt các thư viện cần thiết (không cần cài thêm gì vì chỉ sử dụng thư viện tiêu chuẩn `tkinter` và `collections`).
2. Mở file `BTVN.ipynb` trên VS Code hoặc Jupyter Notebook.
3. Chọn và chạy cell code mong muốn:
   - **Chạy BFS**: Nhấp "Run" ở cell thứ nhất.
   - **Chạy DFS**: Nhấp "Run" ở cell thứ hai.
4. Trên giao diện Tkinter mở ra, bạn có thể:
   - Nhấp **Next Step** để xem từng bước tìm kiếm một cách thủ công.
   - Nhấp **Auto Run** để tự động chạy mô phỏng.
   - Kéo thanh trượt **Speed** để điều chỉnh tốc độ tự động chạy.
   - Sử dụng chuột cuộn ngang trên vùng **Frontier** hoặc **Explored** (hoặc dùng Shift + cuộn chuột) để xem các trạng thái phía sau.


In [ ]:
# =========================================================================
# CHƯƠNG TRÌNH GIẢI BÀI TOÁN 8-PUZZLE BẰNG THUẬT TOÁN BREADTH-FIRST SEARCH (BFS)
# - Sử dụng hàng đợi (Queue - FIFO) để tìm đường đi ngắn nhất.
# - Trực quan hóa thời gian thực (Real-time) từng bước thông qua Tkinter.
# - Hỗ trợ thanh cuộn ngang, điều chỉnh tốc độ (Speed) và tự động chạy (Auto Run).
# =========================================================================

from collections import deque
import tkinter as tk

START_STATE = ((2,8,3),(1,6,4),(7,0,5))
GOAL_STATE  = ((1,2,3),(8,0,4),(7,6,5))
MOVES = [(-1,0,'Up'),(1,0,'Down'),(0,-1,'Left'),(0,1,'Right')]

# ── helpers ──────────────────────────────────────────────
def get_neighbors(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j
    result = []
    for dx, dy, act in MOVES:
        nx, ny = x+dx, y+dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            s = [list(r) for r in state]
            s[x][y], s[nx][ny] = s[nx][ny], s[x][y]
            result.append((tuple(tuple(r) for r in s), act))
    return result

def reconstruct(node):
    path = []
    while node:
        path.append(node['state'])
        node = node['parent']
    return path[::-1]

# ── BFS generator ────────────────────────────────────────
def bfs_gen(start, goal):
    queue = deque([{'state': start, 'parent': None, 'depth': 0}])
    visited = {start}
    expanded = generated = 0
    while queue:
        node = queue.popleft()
        expanded += 1
        frontier = [n['state'] for n in queue]
        yield node['state'], frontier, list(visited), expanded, generated, None
        if node['state'] == goal:
            yield node['state'], [], list(visited), expanded, generated, reconstruct(node)
            return
        for nb, _ in get_neighbors(node['state']):
            if nb not in visited:
                visited.add(nb)
                queue.append({'state': nb, 'parent': node, 'depth': node['depth']+1})
                generated += 1

# ── Scrollable Frame ─────────────────────────────────────
class ScrollableFrame(tk.Frame):
    def __init__(self, parent, height=180):
        super().__init__(parent)
        self.h_scroll = tk.Scrollbar(self, orient='horizontal')
        self.h_scroll.pack(side='bottom', fill='x')
        self.canvas = tk.Canvas(self, height=height)
        self.canvas.pack(side='left', fill='both', expand=True)
        self.h_scroll.config(command=self.canvas.xview)
        self.canvas.configure(xscrollcommand=self.h_scroll.set)
        self.inner = tk.Frame(self.canvas)
        self.inner.bind('<Configure>', lambda e: self.canvas.configure(scrollregion=self.canvas.bbox('all')))
        self.canvas.create_window((0,0), window=self.inner, anchor='nw')
        self.canvas.bind('<Shift-MouseWheel>', lambda e: self.canvas.xview_scroll(int(-1*(e.delta/120)),'units'))

# ── GUI ──────────────────────────────────────────────────
class PuzzleGUI:
    def __init__(self, root, gen, title='BFS'):
        self.root = root
        self.gen  = gen
        self.running = False
        self.step_count = 0
        root.title(f'8 Puzzle {title} – Real-time')
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        w, h = min(1400, int(sw*.85)), min(900, int(sh*.85))
        root.geometry(f'{w}x{h}+{(sw-w)//2}+{(sh-h)//2}')
        root.minsize(900, 600)

        self.info = tk.Label(root, text='Press Auto Run or Next Step', font=('Arial',14,'bold'))
        self.info.pack(pady=6)

        self.cur_frame = tk.LabelFrame(root, text='Current State', padx=8, pady=8)
        self.cur_frame.pack(fill='x', padx=10, pady=4)

        fl = tk.LabelFrame(root, text='Frontier (Head - Rear)', padx=8, pady=8)
        fl.pack(fill='both', expand=True, padx=10, pady=4)
        self.frontier_sf = ScrollableFrame(fl, 180)
        self.frontier_sf.pack(fill='both', expand=True)

        el = tk.LabelFrame(root, text='Explored', padx=8, pady=8)
        el.pack(fill='both', expand=True, padx=10, pady=4)
        self.explored_sf = ScrollableFrame(el, 180)
        self.explored_sf.pack(fill='both', expand=True)

        bf = tk.Frame(root)
        bf.pack(pady=8)
        tk.Button(bf, text='Next Step', width=14, command=self.next_step).pack(side='left', padx=5)
        self.auto_btn = tk.Button(bf, text='Auto Run', width=14, command=self.toggle_auto)
        self.auto_btn.pack(side='left', padx=5)

        self.speed_var = tk.IntVar(value=300)
        tk.Label(bf, text='Speed (ms):').pack(side='left', padx=(15,2))
        tk.Scale(bf, from_=10, to=1000, orient='horizontal', variable=self.speed_var, length=160).pack(side='left')

    def draw_state(self, parent, state):
        f = tk.Frame(parent)
        for i in range(3):
            for j in range(3):
                v = state[i][j]
                tk.Label(f, text=str(v) if v else '', width=4, height=2,
                         font=('Arial',13,'bold'), relief='solid',
                         bg='#4CAF50' if v == 0 else 'white').grid(row=i, column=j)
        return f

    def clear(self, frame):
        for w in frame.winfo_children(): w.destroy()

    def render(self, current, frontier, explored, expanded, generated, path):
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)
        self.clear(self.explored_sf.inner)

        self.draw_state(self.cur_frame, current).pack()
        for i, s in enumerate(frontier):
            self.draw_state(self.frontier_sf.inner, s).grid(row=0, column=i, padx=8, pady=8)
        for i, s in enumerate(explored):
            self.draw_state(self.explored_sf.inner, s).grid(row=0, column=i, padx=8, pady=8)

        status = f'Step {self.step_count} | Expanded: {expanded} | Generated: {generated}'
        if path:
            status += f' | GOAL FOUND! Path length: {len(path)-1}'
        self.info.config(text=status)

    def next_step(self):
        try:
            current, frontier, explored, expanded, generated, path = next(self.gen)
            self.step_count += 1
            self.render(current, frontier, explored, expanded, generated, path)
            if path:
                self.running = False
                self.auto_btn.config(text='Auto Run')
        except StopIteration:
            self.running = False
            self.auto_btn.config(text='Done')

    def toggle_auto(self):
        self.running = not self.running
        self.auto_btn.config(text='Pause' if self.running else 'Auto Run')
        if self.running:
            self.auto_tick()

    def auto_tick(self):
        if not self.running: return
        self.next_step()
        if self.running:
            self.root.after(self.speed_var.get(), self.auto_tick)

root = tk.Tk()
app = PuzzleGUI(root, bfs_gen(START_STATE, GOAL_STATE), 'BFS')
root.mainloop()

In [ ]:
# =========================================================================
# CHƯƠNG TRÌNH GIẢI BÀI TOÁN 8-PUZZLE BẰNG THUẬT TOÁN DEPTH-FIRST SEARCH (DFS)
# - Sử dụng ngăn xếp (Stack - LIFO) để đi sâu nhất có thể trên mỗi nhánh.
# - Trực quan hóa thời gian thực (Real-time) từng bước giống hệt BFS.
# - Hỗ trợ thanh cuộn ngang, điều chỉnh tốc độ (Speed) và tự động chạy (Auto Run).
# =========================================================================

import tkinter as tk

START_STATE = ((2,8,3),(1,6,4),(7,0,5))
GOAL_STATE  = ((1,2,3),(8,0,4),(7,6,5))
MOVES = [(-1,0,'Up'),(1,0,'Down'),(0,-1,'Left'),(0,1,'Right')]

def get_neighbors(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j
    result = []
    for dx, dy, act in MOVES:
        nx, ny = x+dx, y+dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            s = [list(r) for r in state]
            s[x][y], s[nx][ny] = s[nx][ny], s[x][y]
            result.append((tuple(tuple(r) for r in s), act))
    return result

def reconstruct(node):
    path = []
    while node:
        path.append(node['state'])
        node = node['parent']
    return path[::-1]

# ── DFS generator ────────────────────────────────────────
def dfs_gen(start, goal):
    stack = [{'state': start, 'parent': None, 'depth': 0}]
    visited = set()
    expanded = generated = 0
    while stack:
        node = stack.pop()
        if node['state'] in visited:
            continue
        visited.add(node['state'])
        expanded += 1
        frontier = [n['state'] for n in stack]
        yield node['state'], frontier, list(visited), expanded, generated, None
        if node['state'] == goal:
            yield node['state'], [], list(visited), expanded, generated, reconstruct(node)
            return
        for nb, _ in reversed(get_neighbors(node['state'])):
            if nb not in visited:
                stack.append({'state': nb, 'parent': node, 'depth': node['depth']+1})
                generated += 1

class ScrollableFrame(tk.Frame):
    def __init__(self, parent, height=180):
        super().__init__(parent)
        self.h_scroll = tk.Scrollbar(self, orient='horizontal')
        self.h_scroll.pack(side='bottom', fill='x')
        self.canvas = tk.Canvas(self, height=height)
        self.canvas.pack(side='left', fill='both', expand=True)
        self.h_scroll.config(command=self.canvas.xview)
        self.canvas.configure(xscrollcommand=self.h_scroll.set)
        self.inner = tk.Frame(self.canvas)
        self.inner.bind('<Configure>', lambda e: self.canvas.configure(scrollregion=self.canvas.bbox('all')))
        self.canvas.create_window((0,0), window=self.inner, anchor='nw')
        self.canvas.bind('<Shift-MouseWheel>', lambda e: self.canvas.xview_scroll(int(-1*(e.delta/120)),'units'))

class PuzzleGUI:
    def __init__(self, root, gen, title='DFS'):
        self.root = root
        self.gen  = gen
        self.running = False
        self.step_count = 0
        root.title(f'8 Puzzle {title} – Real-time')
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        w, h = min(1400, int(sw*.85)), min(900, int(sh*.85))
        root.geometry(f'{w}x{h}+{(sw-w)//2}+{(sh-h)//2}')
        root.minsize(900, 600)

        self.info = tk.Label(root, text='Press Auto Run or Next Step', font=('Arial',14,'bold'))
        self.info.pack(pady=6)

        self.cur_frame = tk.LabelFrame(root, text='Current State', padx=8, pady=8)
        self.cur_frame.pack(fill='x', padx=10, pady=4)

        fl = tk.LabelFrame(root, text='Frontier (Bottom - Top)', padx=8, pady=8)
        fl.pack(fill='both', expand=True, padx=10, pady=4)
        self.frontier_sf = ScrollableFrame(fl, 180)
        self.frontier_sf.pack(fill='both', expand=True)

        el = tk.LabelFrame(root, text='Explored', padx=8, pady=8)
        el.pack(fill='both', expand=True, padx=10, pady=4)
        self.explored_sf = ScrollableFrame(el, 180)
        self.explored_sf.pack(fill='both', expand=True)

        bf = tk.Frame(root)
        bf.pack(pady=8)
        tk.Button(bf, text='Next Step', width=14, command=self.next_step).pack(side='left', padx=5)
        self.auto_btn = tk.Button(bf, text='Auto Run', width=14, command=self.toggle_auto)
        self.auto_btn.pack(side='left', padx=5)

        self.speed_var = tk.IntVar(value=300)
        tk.Label(bf, text='Speed (ms):').pack(side='left', padx=(15,2))
        tk.Scale(bf, from_=10, to=1000, orient='horizontal', variable=self.speed_var, length=160).pack(side='left')

    def draw_state(self, parent, state):
        f = tk.Frame(parent)
        for i in range(3):
            for j in range(3):
                v = state[i][j]
                tk.Label(f, text=str(v) if v else '', width=4, height=2,
                         font=('Arial',13,'bold'), relief='solid',
                         bg='#4CAF50' if v == 0 else 'white').grid(row=i, column=j)
        return f

    def clear(self, frame):
        for w in frame.winfo_children(): w.destroy()

    def render(self, current, frontier, explored, expanded, generated, path):
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)
        self.clear(self.explored_sf.inner)

        self.draw_state(self.cur_frame, current).pack()
        for i, s in enumerate(frontier):
            self.draw_state(self.frontier_sf.inner, s).grid(row=0, column=i, padx=8, pady=8)
        for i, s in enumerate(explored):
            self.draw_state(self.explored_sf.inner, s).grid(row=0, column=i, padx=8, pady=8)

        status = f'Step {self.step_count} | Expanded: {expanded} | Generated: {generated}'
        if path:
            status += f' | GOAL FOUND! Path length: {len(path)-1}'
        self.info.config(text=status)

    def next_step(self):
        try:
            current, frontier, explored, expanded, generated, path = next(self.gen)
            self.step_count += 1
            self.render(current, frontier, explored, expanded, generated, path)
            if path:
                self.running = False
                self.auto_btn.config(text='Auto Run')
        except StopIteration:
            self.running = False
            self.auto_btn.config(text='Done')

    def toggle_auto(self):
        self.running = not self.running
        self.auto_btn.config(text='Pause' if self.running else 'Auto Run')
        if self.running:
            self.auto_tick()

    def auto_tick(self):
        if not self.running: return
        self.next_step()
        if self.running:
            self.root.after(self.speed_var.get(), self.auto_tick)

root = tk.Tk()
app = PuzzleGUI(root, dfs_gen(START_STATE, GOAL_STATE), 'DFS')
root.mainloop()